# The CUDA Programming Model

Master the CUDA thread hierarchy: kernels, threads, blocks, and grids. Learn to write and launch GPU kernels with numba.cuda, calculate global thread indices, and handle 2D indexing for matrix and image operations.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-programming-model)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Kernels, Threads, and Grids

In [ ]:
!pip install numba

import numpy as np
from numba import cuda

# ============================================
# Understanding the CUDA Thread Hierarchy
# ============================================

# Kernel that reports its own thread/block indices
@cuda.jit
def identify_threads(output):
    tid = cuda.threadIdx.x      # Thread index within block
    bid = cuda.blockIdx.x       # Block index within grid
    bdim = cuda.blockDim.x      # Threads per block
    global_id = bid * bdim + tid  # Same as cuda.grid(1)

    if global_id < output.shape[0]:
        # Pack block_id, thread_id, global_id into output
        output[global_id, 0] = bid
        output[global_id, 1] = tid
        output[global_id, 2] = global_id

# Launch with 3 blocks of 4 threads (small, for illustration)
n_threads_total = 12
output = np.zeros((n_threads_total, 3), dtype=np.int32)
d_output = cuda.to_device(output)

threads_per_block = 4
blocks_per_grid = 3
identify_threads[blocks_per_grid, threads_per_block](d_output)
result = d_output.copy_to_host()

print("=== Thread Hierarchy (3 blocks x 4 threads) ===")
print(f"{'Global ID':>10} | {'Block ID':>8} | {'Thread ID':>9}")
print("-" * 35)
for i in range(n_threads_total):
    print(f"{int(result[i, 2]):>10} | {int(result[i, 0]):>8} | {int(result[i, 1]):>9}")

# ============================================
# Grid size calculation: ceiling division
# ============================================
print("\n=== Grid Size Calculation ===")
data_sizes = [100, 1000, 1_000_000, 10_000_000]
for n in data_sizes:
    tpb = 256
    bpg = (n + tpb - 1) // tpb
    total_threads = bpg * tpb
    wasted = total_threads - n
    print(f"  n={n:>10,}: {bpg:>6,} blocks x {tpb} threads = {total_threads:>10,} total ({wasted:>5} idle)")

# ============================================
# Complete example: double every element
# ============================================
print("\n=== Double Elements Kernel ===")

@cuda.jit
def double_elements(arr):
    idx = cuda.grid(1)
    if idx < arr.size:
        arr[idx] *= 2

n = 1_000_000
data = np.arange(n, dtype=np.float32)
d_data = cuda.to_device(data)

tpb = 256
bpg = (n + tpb - 1) // tpb

double_elements[bpg, tpb](d_data)
result = d_data.copy_to_host()

print(f"Input:  [0, 1, 2, ..., {n-1}]")
print(f"Output: [{result[0]:.0f}, {result[1]:.0f}, {result[2]:.0f}, ..., {result[-1]:.0f}]")
assert np.allclose(result, np.arange(n, dtype=np.float32) * 2)
print("Correctness verified!")

### Lesson example: Your First Kernel: Vector Addition

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda

# ============================================
# Step-by-step Vector Addition
# ============================================

# Step 1: Define the kernel
@cuda.jit
def vector_add(a, b, c):
    idx = cuda.grid(1)
    if idx < a.size:
        c[idx] = a[idx] + b[idx]

# Step 2: Prepare data on CPU
n = 10_000_000
print(f"Vector size: {n:,} elements ({n * 4 / 1e6:.1f} MB per array)")
a = np.random.randn(n).astype(np.float32)
b = np.random.randn(n).astype(np.float32)

# Step 3: Transfer to GPU
print("\nTransferring data to GPU...")
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.device_array(n, dtype=np.float32)
print(f"  d_a type: {type(d_a)}, shape: {d_a.shape}, dtype: {d_a.dtype}")

# Step 4: Calculate launch configuration
threads_per_block = 256
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block
print(f"\nLaunch config: {blocks_per_grid:,} blocks x {threads_per_block} threads")
print(f"  Total threads: {blocks_per_grid * threads_per_block:,}")
print(f"  Extra threads: {blocks_per_grid * threads_per_block - n:,} (idle, handled by boundary check)")

# Step 5: Launch! (first call compiles the kernel)
print("\nFirst launch (includes compilation)...")
start = time.perf_counter()
vector_add[blocks_per_grid, threads_per_block](d_a, d_b, d_c)
cuda.synchronize()
first_time = time.perf_counter() - start
print(f"  Time: {first_time*1000:.1f} ms (includes JIT compilation)")

# Subsequent launches reuse compiled kernel
print("\nSubsequent launches (pre-compiled)...")
times = []
for i in range(10):
    start = time.perf_counter()
    vector_add[blocks_per_grid, threads_per_block](d_a, d_b, d_c)
    cuda.synchronize()
    times.append(time.perf_counter() - start)
print(f"  Average: {np.mean(times)*1000:.3f} ms")
print(f"  Speedup vs first call: {first_time / np.mean(times):.0f}x")

# Step 6: Retrieve and verify
c_gpu = d_c.copy_to_host()
c_cpu = a + b
assert np.allclose(c_gpu, c_cpu, atol=1e-5)
print(f"\nResults verified! First 5 elements:")
print(f"  CPU: {c_cpu[:5]}")
print(f"  GPU: {c_gpu[:5]}")

# ============================================
# Compare with CPU (NumPy)
# ============================================
print("\n=== Performance Comparison ===")

# NumPy (CPU)
start = time.perf_counter()
for _ in range(10):
    c_cpu = a + b
cpu_time = (time.perf_counter() - start) / 10

# GPU (compute only, data already on device)
start = time.perf_counter()
for _ in range(10):
    vector_add[blocks_per_grid, threads_per_block](d_a, d_b, d_c)
cuda.synchronize()
gpu_time = (time.perf_counter() - start) / 10

print(f"NumPy (CPU): {cpu_time*1000:.3f} ms")
print(f"CUDA (GPU):  {gpu_time*1000:.3f} ms")
print(f"Speedup:     {cpu_time/gpu_time:.1f}x")

### Lesson example: 2D Thread Indexing

In [ ]:
!pip install numba

import numpy as np
import time
from numba import cuda

# ============================================
# 2D Thread Indexing Demonstration
# ============================================

# Small example: visualize 2D thread mapping
@cuda.jit
def map_threads_2d(output):
    col, row = cuda.grid(2)
    if row < output.shape[0] and col < output.shape[1]:
        # Store the global thread index at each position
        bid_x = cuda.blockIdx.x
        bid_y = cuda.blockIdx.y
        tid_x = cuda.threadIdx.x
        tid_y = cuda.threadIdx.y
        # Encode block and thread info
        output[row, col, 0] = row
        output[row, col, 1] = col
        output[row, col, 2] = bid_x * 100 + bid_y  # block (x,y)
        output[row, col, 3] = tid_x * 100 + tid_y  # thread (x,y)

# 6x8 matrix, 4x4 blocks
H, W = 6, 8
output = np.zeros((H, W, 4), dtype=np.int32)
d_output = cuda.to_device(output)

TPB = (4, 4)
BPG = ((W + 3) // 4, (H + 3) // 4)  # (cols, rows)
print(f"Grid: {BPG[0]}x{BPG[1]} blocks, each {TPB[0]}x{TPB[1]} threads")

map_threads_2d[BPG, TPB](d_output)
result = d_output.copy_to_host()

print(f"\n=== 2D Thread Mapping ({H}x{W} matrix) ===")
print("Position (row,col) -> Block(bx,by), Thread(tx,ty)")
for r in range(H):
    for c in range(W):
        bxy = result[r, c, 2]
        txy = result[r, c, 3]
        bx, by = bxy // 100, bxy % 100
        tx, ty = txy // 100, txy % 100
        print(f"  ({r},{c})->B({bx},{by})T({tx},{ty})", end="")
    print()

# ============================================
# Practical: Element-wise matrix operations
# ============================================
print("\n=== Element-wise Matrix Operations ===")

@cuda.jit
def matrix_multiply_elementwise(A, B, C):
    col, row = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] * B[row, col]

@cuda.jit
def matrix_relu(input_mat, output_mat):
    col, row = cuda.grid(2)
    if row < input_mat.shape[0] and col < input_mat.shape[1]:
        val = input_mat[row, col]
        output_mat[row, col] = val if val > 0 else 0.0

# Large matrix operations
M, N = 4096, 4096
A = np.random.randn(M, N).astype(np.float32)
B = np.random.randn(M, N).astype(np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((M, N), dtype=np.float32)
d_relu = cuda.device_array((M, N), dtype=np.float32)

TPB = (16, 16)
BPG = ((N + TPB[0] - 1) // TPB[0], (M + TPB[1] - 1) // TPB[1])
print(f"Matrix: {M}x{N}, Blocks: {BPG}, Threads/block: {TPB}")

# Benchmark element-wise multiply
matrix_multiply_elementwise[BPG, TPB](d_A, d_B, d_C)  # Warm up
cuda.synchronize()

start = time.perf_counter()
for _ in range(100):
    matrix_multiply_elementwise[BPG, TPB](d_A, d_B, d_C)
cuda.synchronize()
gpu_time = (time.perf_counter() - start) / 100

start = time.perf_counter()
for _ in range(100):
    C_cpu = A * B
cpu_time = (time.perf_counter() - start) / 100

C_gpu = d_C.copy_to_host()
assert np.allclose(C_gpu, A * B, atol=1e-5)
print(f"\nElement-wise multiply ({M}x{N}):")
print(f"  NumPy (CPU): {cpu_time*1000:.3f} ms")
print(f"  CUDA (GPU):  {gpu_time*1000:.3f} ms")
print(f"  Speedup:     {cpu_time/gpu_time:.1f}x")

# Benchmark ReLU
matrix_relu[BPG, TPB](d_A, d_relu)  # Warm up
cuda.synchronize()

start = time.perf_counter()
for _ in range(100):
    matrix_relu[BPG, TPB](d_A, d_relu)
cuda.synchronize()
gpu_relu_time = (time.perf_counter() - start) / 100

start = time.perf_counter()
for _ in range(100):
    relu_cpu = np.maximum(A, 0)
cpu_relu_time = (time.perf_counter() - start) / 100

relu_gpu = d_relu.copy_to_host()
assert np.allclose(relu_gpu, np.maximum(A, 0), atol=1e-5)
print(f"\nReLU ({M}x{N}):")
print(f"  NumPy (CPU): {cpu_relu_time*1000:.3f} ms")
print(f"  CUDA (GPU):  {gpu_relu_time*1000:.3f} ms")
print(f"  Speedup:     {cpu_relu_time/gpu_relu_time:.1f}x")

---

## Exercise: Element-wise Matrix Operations with 2D Indexing


In [ ]:
import numpy as np
import math
import time
from numba import cuda

# TODO: Implement matrix_add kernel
@cuda.jit
def matrix_add(A, B, C):
    pass

# TODO: Implement matrix_scale kernel
@cuda.jit
def matrix_scale(A, scalar, C):
    pass

# TODO: Implement matrix_sigmoid kernel
# Hint: use math.exp() for the exponential
@cuda.jit
def matrix_sigmoid(A, C):
    pass

# TODO: Implement matrix_clamp kernel
@cuda.jit
def matrix_clamp(A, lo, hi, C):
    pass

def launch_2d(kernel, args, M, N):
    """Helper to launch a 2D kernel with proper configuration."""
    TPB = (16, 16)
    # TODO: Calculate BPG correctly (remember: x=cols, y=rows)
    BPG = (1, 1)  # Fix this!
    kernel[BPG, TPB](*args)
    cuda.synchronize()

# Test with NON-SQUARE matrix to catch row/col bugs
M, N = 2000, 3000  # 2000 rows, 3000 columns
A = np.random.randn(M, N).astype(np.float32)
B = np.random.randn(M, N).astype(np.float32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((M, N), dtype=np.float32)

# TODO: Test each kernel
# TODO: Verify against NumPy
# TODO: Benchmark sigmoid (4096x4096) GPU vs CPU

## Your tasks

Implement several element-wise matrix operations using 2D CUDA kernels, verify correctness, and benchmark against NumPy.

**Requirements:**
1. Implement these GPU kernels with proper 2D indexing:
   - `matrix_add(A, B, C)`: C = A + B
   - `matrix_scale(A, scalar, C)`: C = A * scalar
   - `matrix_sigmoid(A, C)`: C = 1 / (1 + exp(-A))
   - `matrix_clamp(A, lo, hi, C)`: C = clamp each element to [lo, hi]
2. Each kernel must:
   - Use `col, row = cuda.grid(2)` for 2D indexing
   - Include proper boundary checks
   - Work with arbitrary matrix sizes (not just multiples of block size)
3. Test with a non-square matrix (e.g., 2000 x 3000) to catch row/col bugs
4. Verify each kernel's output against the NumPy equivalent
5. Benchmark sigmoid on a 4096x4096 matrix: GPU vs NumPy

**Hints:**
- For sigmoid, use `math.exp()` inside the kernel (import math at module level)
- Clamp: `min(max(val, lo), hi)`
- blocks_per_grid = ((N + TPB_x - 1) // TPB_x, (M + TPB_y - 1) // TPB_y) where N=cols, M=rows
- Test with non-square matrices to catch the (x,y) vs (row,col) confusion

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-programming-model) for the solution and discussion.